# Exploratory Data Analysis

In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from category_encoders import TargetEncoder
import pandas as pd
# pip install pyarrow

### Objective: Next-Day Return Prediction

- Goal: Train a machine learning model to predict whether a stock will increase by at least 1% on the next trading day, using information from the S&P 500 and its constituent companies.

- Description: The model leverages technical indicators, price action, volume dynamics, volatility measures, and lagged features to identify short-term market patterns that precede positive price movements over a one-day horizon. The task is framed as a binary classification problem, where the target variable indicates whether the next-day return exceeds the 1% threshold.

- Purpose: To evaluate the feasibility of very short-term return prediction using historical market data while strictly avoiding look-ahead bias and preserving the temporal structure of financial time series.

### Data Origin and Description

The data for this project comes from Kaggle, specifically the Advanced Stock Dataset by baidalinadilzhan, which is built using historical data retrieved from the Yahoo Finance API and covers approximately the past five years of trading for companies in the S&P 500 index. The dataset is structured with daily adjusted prices that account for corporate actions such as dividends and stock splits, making it suitable for financial analysis and time-series modeling.

The full dataset includes over 620,000 daily observations with 73 engineered features, including opening, high, low, and closing prices, trading volume, technical indicators (e.g., moving averages, RSI), volatility measures, and multiple lagged variables. These features provide a rich foundation for short-term return forecasting, enabling the construction of a binary classification target based on the next-day return, which indicates whether a stock achieves a return of at least 1% on the following trading day. All targets are derived using strictly forward-looking returns, ensuring the preservation of temporal structure and the avoidance of look-ahead bias.

### Dataset Columns
- DATE: Trading date.
- TICKER: Stock ticker symbol.
- OPEN: Opening price of the day.
- HIGH: Highest price during the day.
- LOW: Lowest price during the day.
- CLOSE: Closing price of the day.
- VOLUME: Total number of shares traded.
- DIVIDENDS: Dividend payments on that date (if any).
- STOCK_SPLITS: Stock split events on that date (if any).
- SMA_5: 5-day simple moving average of the closing price.
- SMA_10: 10-day simple moving average.
- SMA_20: 20-day simple moving average.
- SMA_50: 50-day simple moving average.
- EMA_12: 12-day exponential moving average.
- EMA_26: 26-day exponential moving average.
- MACD: Difference between EMA_12 and EMA_26 (momentum indicator).
- MACD_SIGNAL: Moving average of the MACD.
- MACD_HISTOGRAM: Difference between MACD and its signal line.
- RSI: Relative Strength Index (14 periods).
- VOLATILITY: Historical price volatility.
- BB_MIDDLE: Middle Bollinger Band (moving average).
- BB_UPPER: Upper Bollinger Band.
- BB_LOWER: Lower Bollinger Band.
- BB_WIDTH: Bollinger Band width (relative volatility measure).
- BB_POSITION: Price position within Bollinger Bands.
- PRICE_CHANGE: Daily closing price change.
- PRICE_CHANGE_5D: Cumulative price change over the last 5 days.
- HIGH_LOW_RATIO: Ratio between high and low prices.
- OPEN_CLOSE_RATIO: Ratio between open and close prices.
- VOLUME_SMA: Moving average of volume.
- VOLUME_RATIO: Current volume relative to its moving average.
- CLOSE_LAG_1: Closing price 1 day ago.
- CLOSE_LAG_2: Closing price 2 days ago.
- CLOSE_LAG_3: Closing price 3 days ago.
- CLOSE_LAG_4: Closing price 4 days ago.
- CLOSE_LAG_5: Closing price 5 days ago.
- CLOSE_LAG_6: Closing price 6 days ago.
- CLOSE_LAG_7: Closing price 7 days ago.
- CLOSE_LAG_8: Closing price 8 days ago.
- CLOSE_LAG_9: Closing price 9 days ago.
- CLOSE_LAG_10: Closing price 10 days ago.
- VOLUME_LAG_1: Volume 1 day ago.
- VOLUME_LAG_2: Volume 2 days ago.
- VOLUME_LAG_3: Volume 3 days ago.
- VOLUME_LAG_4: Volume 4 days ago.
- VOLUME_LAG_5: Volume 5 days ago.
- VOLUME_LAG_6: Volume 6 days ago.
- VOLUME_LAG_7: Volume 7 days ago.
- VOLUME_LAG_8: Volume 8 days ago.
- VOLUME_LAG_9: Volume 9 days ago.
- VOLUME_LAG_10: Volume 10 days ago.
- PRICE_CHANGE_LAG_1: Price change 1 day ago.
- PRICE_CHANGE_LAG_2: Price change 2 days ago.
- PRICE_CHANGE_LAG_3: Price change 3 days ago.
- PRICE_CHANGE_LAG_4: Price change 4 days ago.
- PRICE_CHANGE_LAG_5: Price change 5 days ago.
- PRICE_CHANGE_LAG_6: Price change 6 days ago.
- PRICE_CHANGE_LAG_7: Price change 7 days ago.
- PRICE_CHANGE_LAG_8: Price change 8 days ago.
- PRICE_CHANGE_LAG_9: Price change 9 days ago.
- PRICE_CHANGE_LAG_10: Price change 10 days ago.
- RSI_LAG_1: RSI 1 day ago.
- RSI_LAG_2: RSI 2 days ago.
- RSI_LAG_3: RSI 3 days ago.
- RSI_LAG_4: RSI 4 days ago.
- RSI_LAG_5: RSI 5 days ago.
- MACD_LAG_1: MACD 1 day ago.
- MACD_LAG_2: MACD 2 days ago.
- MACD_LAG_3: MACD 3 days ago.
- MACD_LAG_4: MACD 4 days ago.
- MACD_LAG_5: MACD 5 days ago.
- VOLATILITY_LAG_1: Volatility 1 day ago.
- VOLATILITY_LAG_2: Volatility 2 days ago.
- FUTURE_RETURN_1D / 5D / 10D / 20D: Future return over N days.
- FUTURE_UP_1D / 5D / 10D / 20D: Binary label (1 if return > 0, else 0).
- FUTURE_CATEGORY_1D / 5D / 10D / 20D: Categorical future return class.

## Load data and create DataFrame

In [ ]:
# Define the path where the files are located
path = r'../data/raw/'

In [ ]:
# List Parquet files starting with 'data'
all_files = glob.glob(os.path.join(path, 'data*.parquet'))

In [ ]:
# Read each Parquet file and store it in a list of DataFrames
df_list = []
for filename in all_files:
    df_part = pd.read_parquet(filename, engine='pyarrow')
    df_list.append(df_part)

In [ ]:
# Concatenate all DataFrames into a single DataFrame
df = pd.concat(df_list, axis=0, ignore_index=True)

In [ ]:
# Convert Date to datetime (required for temporal split)
df['Date'] = pd.to_datetime(df['Date'])

## Descriptive Analysis

In [ ]:
# Show first rows of the dataset
df.head()

In [ ]:
# Check number of rows and columns
df.shape

In [ ]:
# Display the column names of the DataFrame
df.columns

In [ ]:
# Descriptive statistics for numerical variables
df.describe()

In [ ]:
# General DataFrame info
df.info()

In [ ]:
# Count missing values per column
df.isnull().sum().sort_values(ascending=False)

### Observations
- The dataset contains a total of 620,095 rows.
- It has 73 columns in total.
- No duplicate rows were found in the dataset.
- There are no missing values in any column.
- Out of the 73 columns, 71 are numeric.
- Two columns are of object type.

## Data Cleaning

In [ ]:
# Sort by Ticker and Date
df = df.sort_values(['Ticker','Date'])

In [ ]:
# Remove columns that will not be used initially for exploratory analysis or modeling
df = df.drop(['Dividends', 'Stock Splits', 'SMA_10', 'SMA_50', 'MACD_Histogram', 'BB_Middle',
              'BB_Upper', 'BB_Lower', 'BB_Width', 'Price_Change_5d', 'High_Low_Ratio', 'Open_Close_Ratio', 'Volume_SMA',
              'Close_lag_5', 'Close_lag_10','Price_Change_lag_1', 'Price_Change_lag_2',
              'Price_Change_lag_3', 'Price_Change_lag_5', 'Price_Change_lag_10', 'RSI_lag_2', 'RSI_lag_3', 'RSI_lag_5', 'RSI_lag_10',
              'MACD_lag_2', 'MACD_lag_3', 'MACD_lag_5', 'MACD_lag_10', 'Future_Category_1d', 'Future_Return_5d',
              'Future_Up_5d', 'Future_Category_5d', 'Future_Return_10d','Future_Up_10d', 'Future_Category_10d', 
              'Future_Return_20d', 'Future_Up_20d', 'Future_Category_20d','Future_Return_1d'], axis=1)

### Observations
Columns that did not provide relevant information for the main analysis and modeling objective were removed, such as long-term metrics, redundant indicators, and future returns over 5, 10, and 20 days, in order to simplify the dataset and prevent data leakage. The Future_Up_1d variable was selected as the target because it represents, in binary form, whether a stock will increase by at least 1 percent on the following trading day, directly aligning with the model’s objective.

In [ ]:
# Count how many zero values exist in each column of the DataFrame
value_cero = (df == 0).sum()

In [ ]:
# Display only the columns that contain at least one zero
value_cero[value_cero > 0]

In [ ]:
# Check the distribution of values in the 'Price_Change' column
df['Price_Change'].value_counts()

### Observations
We check the number of zeros in the dataset to ensure they do not indicate errors or missing data. Although Price_Change has several zeros, this is expected and consistent with actual price movements, so they are not considered anomalies.

## Visualization

In [ ]:
# Create a DataFrame with only numeric columns (excluding 'Ticker'and 'Date')
df_filtrado = df.drop(columns=['Ticker','Date'])

# Compute the correlation matrix for numeric variables
corr_numerical = df_filtrado.corr(numeric_only=True)

# Filter strong correlations and prepare the upper-triangle mask
high_correlation = corr_numerical[corr_numerical.abs() > 0.40]
mask = np.triu(np.ones_like(corr_numerical, dtype=bool))

# Plot the heatmap of correlations
fig, axis = plt.subplots(figsize=(15,15))
sns.heatmap(high_correlation, annot=True, mask=mask, linewidths=3, fmt='.2f')

plt.tight_layout()
plt.show()

In [ ]:
# Remove highly redundant columns identified in the correlation heatmap
cols_to_drop = ['Open', 'High', 'Low', 'SMA_5', 'SMA_20', 'EMA_26', 'Close_lag_1', 'Close_lag_2',
                'Close_lag_3', 'Volatility_lag_2', 'Volatility_lag_3', 'Volatility_lag_5', 'Volatility_lag_10']

df = df.drop(columns=cols_to_drop)

### Observations
These columns were removed because they showed very high correlations (0.85 or higher) with other variables, indicating redundancy and potential multicollinearity. None of them had a direct relationship with the target Future_Up_1d, so dropping them simplifies the dataset without losing relevant information for modeling.

In [ ]:
# Check how the target 'Future_Up_1d' relates to next day's price change
verificacion = df[['Ticker', 'Price_Change', 'Future_Up_1d','Date']].copy()
verificacion['Next_Day_Change'] = verificacion.groupby('Ticker')['Price_Change'].shift(-1)

In [ ]:
# Calculate the average change when the target is 1
verificacion[verificacion['Future_Up_1d'] == 1]['Next_Day_Change'].mean()

### Conclusion
The results show that the average change when Future_Up_1d is 1 is approximately 0.014 (1.41%). This confirms that our target variable correctly captures the significant upward movements we aim to predict.

## Split Temporal

In [ ]:
# Define the cutoff date (oldest 80 percent for training)
split_date = df['Date'].quantile(0.8)

In [ ]:
# Create temporal datasets
train_df = df[df['Date'] <= split_date]
test_df  = df[df['Date'] > split_date]

In [ ]:
# Define X and y, and drop Date (not used in the model)
X_train = train_df.drop(columns=['Future_Up_1d','Date'])
y_train = train_df['Future_Up_1d']

X_test = test_df.drop(columns=['Future_Up_1d','Date'])
y_test = test_df['Future_Up_1d']

## Target Encoding

Target Encoding was used for the Ticker column because it is a high-cardinality categorical variable with more than 500 distinct categories. Applying One-Hot Encoding would have created hundreds of additional features, unnecessarily increasing the dimensionality of the dataset, the computational cost, and the risk of overfitting. Target Encoding represents each ticker with a single numerical value based on its relationship with the target variable, preserving relevant information without expanding the feature space. This makes the model more efficient, stable, and scalable.

In [ ]:
# Separate numerical features from the categorical 'Ticker' variable
X_train_num = X_train.drop(columns=['Ticker'])
X_test_num = X_test.drop(columns=['Ticker'])

# Apply Target Encoding to the categorical variable
target_encoder = TargetEncoder(cols=['Ticker'])

# Fit the encoder using only the training data
X_train_ticker_encoded = target_encoder.fit_transform(X_train[['Ticker']], y_train)

# Transform the test set using the already fitted encoder
X_test_ticker_encoded = target_encoder.transform(X_test[['Ticker']])

# Combine the original numerical features with the encoded ticker
X_train_final = pd.concat([X_train_num, X_train_ticker_encoded], axis=1)
X_test_final = pd.concat([X_test_num, X_test_ticker_encoded], axis=1)

## Standard Scaler

In [ ]:
# Standard Scaler
scaler = StandardScaler()

# Fit using the complete dataset (numerical features plus encoded ticker)
scaler.fit(X_train_final)

# Convert back to DataFrame
X_train_scaled_array = scaler.transform(X_train_final)
X_test_scaled_array  = scaler.transform(X_test_final)

# Convert back to a DataFrame to preserve column order and names
X_train_scaled = pd.DataFrame(X_train_scaled_array, columns=X_train_final.columns, index=X_train_final.index)
X_test_scaled = pd.DataFrame(X_test_scaled_array, columns=X_test_final.columns, index=X_test_final.index)

## Save Train and Test data

In [ ]:
# Save the processed training and testing datasets (features and target) as CSV files
X_train_scaled.to_csv("../data/processed/X_train_eda2.csv", index = False)
X_test_scaled.to_csv("../data/processed/X_test_eda2.csv", index = False)
y_train.to_csv("../data/processed/y_train_eda2.csv", index = False)
y_test.to_csv("../data/processed/y_test_eda2.csv", index = False)